If you found this tutorial insightful you will also love my work at https://blog.nnitiwe.io

# Set up the environment.

In [ ]:
#mongodb+srv://<db_username>:<db_password>@rag.aatlwy3.mongodb.net/?appName=RAG

In [ ]:
!pip install --quiet --upgrade pymongo voyageai openai langchain langchain-text-splitters langchain_community pypdf

1. Visit https://account.mongodb.com/account/register?nds=true and create an account.

2. Goto **Search & Vector Search** and create a Cluster

3. Under **Security** go to **Database & Network**, click **Database Users**  and create new user.


3. Under **Security** go to **Database & Network**, click **Network Access>> IP Access List** add IP **0.0.0.0/0** (not best practice security reasons).

5. Under **Services**, click **AI Models** and create an VOYAGE API KEY.


6. Visit https://platform.openai.com to create and account and https://platform.openai.com/api-keys to generate OPENAI API keys.


All though you will not be charged... you need to add a card to access the Voyage Model

In [ ]:
import os
from google.colab import userdata


os.environ["VOYAGE_API_KEY"] = userdata.get('VOYAGE_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["RAG_MONGODB_URI"] = userdata.get('RAG_MONGODB_URI')

# Ingest Data (setting up the Retriever)

1. Load Document
2. Split into Chunks
3. Generate Vector Embeddings
4. Store Embeddings

Define a function to generate vector embeddings.

In [ ]:
import os
import voyageai

# Specify the embedding model
model = "voyage-finance-2"
vo = voyageai.Client()

# Define a function to generate embeddings
def get_embedding(data, input_type = "document"):
  embeddings = vo.embed(
      data, model = model, input_type = input_type
  ).embeddings
  return embeddings[0]

Load and split the data.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF
loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

# Split the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(data)

In [ ]:
documents[0]

Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue')

In [ ]:
documents[1]

Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='NEW YORK, May 30, 2024 /PRNewswire/ -- MongoDB, Inc. (NASDAQ: MDB) today announced its financial results for the first quarter ended April 30,\n2024.\n\xa0\n  \xa0\n"MongoDB\'s delivered solid first quarter results, highlighted by 32% Atlas revenue growth. At the same time, we had a slower than expected start to')

In [ ]:
len(documents)

88

Convert the data to vector embeddings.

In [ ]:
# Prepare documents for insertion
docs_to_insert = [{
    "text": doc.page_content,
    "embedding": get_embedding(doc.page_content)
} for doc in documents]

In [ ]:
docs_to_insert[0]

{'text': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue',
 'embedding': [-0.026457443833351135,
  -0.013112200424075127,
  -0.04097260907292366,
  0.02181399054825306,
  0.04991312325000763,
  -0.0176341962069273,
  0.007374300621449947,
  -0.050075508654117584,
  0.025436045601963997,
  0.04571017995476723,
  0.048575565218925476,
  -0.031426649540662766,
  0.012237325310707092,
  -0.032117243856191635,
  -0.03980057314038277,
  0.0014212168753147125,
  0.04119398444890976,
  -0.04937982186675072,
  0.016435762867331505,
  -0.03419290482997894,
  -0.013550516217947006,
  -0.006881421897560358,
  -0.006403113249689341,
  -0.016649065539240837,
  0.004768984392285347,
  -0.051332030445337296,
  0.02796153910458088,
  0.00

Store the data and embeddings in MongoDB.

In [ ]:
from pymongo import MongoClient

# Connect to your MongoDB deployment
client = MongoClient(os.environ["RAG_MONGODB_URI"])
collection = client["RAG"]["semantic_search_table"]

# Insert documents into the collection
result = collection.insert_many(docs_to_insert)

# Use MongoDB Vector Search to retrieve documents.

Create a MongoDB Vector Search index on your vector embeddings.

In [ ]:
from pymongo.operations import SearchIndexModel
import time

# Create your index model, then create the search index
index_name="vector_index"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 1024,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)
collection.create_search_index(model=search_index_model)

# Wait for initial sync to complete
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


Define a function to run vector search queries.

In [ ]:
# Define a function to run vector search queries
def get_query_results(query):
  """Gets results from a vector search query."""

  query_embedding = get_embedding(query, input_type="query")
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index",
              "queryVector": query_embedding,
              "path": "embedding",
              "exact": True,
              "limit": 5
            }
      }, {
            "$project": {
              "_id": 0,
              "text": 1
         }
      }
  ]

  results = collection.aggregate(pipeline)

  array_of_results = []
  for doc in results:
      array_of_results.append(doc)
  return array_of_results

# Test the function with a sample query
import pprint
pprint.pprint(get_query_results("AI technology"))

[{'text': 'MongoDB continues to expand its AI ecosystem with the announcement '
          'of the MongoDB AI Applications Program (MAAP),'},
 {'text': 'artificial intelligence, in our offerings or partnerships; the '
          'growth and expansion of the market for database products and our '
          'ability to penetrate that\n'
          'market; our ability to integrate acquired businesses and '
          'technologies successfully or achieve the expected benefits of such '
          'acquisitions; our ability to'},
 {'text': 'which provides customers with reference architectures, pre-built '
          'partner integrations, and professional services to help\n'
          'them quickly build AI-powered applications. Accenture will '
          'establish a center of excellence focused on MongoDB projects,\n'
          'and is the first global systems integrator to join MAAP.'},
 {'text': 'Bendigo and Adelaide Bank partnered with MongoDB to modernize their '
          'core banking 

# Generate responses with the LLM.

In [ ]:
from openai import OpenAI

# Specify search query, retrieve relevant documents, and convert to string
query = "What are MongoDB's latest AI announcements?"
context_docs = get_query_results(query)
context_string = " ".join([doc["text"] for doc in context_docs])

context_string

'MongoDB continues to expand its AI ecosystem with the announcement of the MongoDB AI Applications Program (MAAP), which provides customers with reference architectures, pre-built partner integrations, and professional services to help\nthem quickly build AI-powered applications. Accenture will establish a center of excellence focused on MongoDB projects,\nand is the first global systems integrator to join MAAP. of MongoDB 8.0—with significant performance improvements such as faster reads and updates, along with significantly\nfaster bulk inserts and time series queries—and the general availability of Atlas Stream Processing to build sophisticated,\nevent-driven applications with real-time data. more of our customers. We also see a tremendous opportunity to win more legacy workloads, as AI has now become a catalyst to modernize these\napplications. MongoDB\'s document-based architecture is particularly well-suited for the variety and scale of data required by AI-powered applications.\x

In [ ]:
# Construct prompt for the LLM using the retrieved documents as the context
prompt = f"""Use the following pieces of context to answer the question at the end.
    {context_string}
    Question: {query}
"""

openai_client = OpenAI()

# OpenAI model to use
model_name = "gpt-4o"

completion = openai_client.chat.completions.create(
model=model_name,
messages=[{"role": "user",
    "content": prompt
  }]
)
print(completion.choices[0].message.content)

MongoDB's latest AI announcements include the introduction of the MongoDB AI Applications Program (MAAP), which provides customers with reference architectures, pre-built partner integrations, and professional services to help them quickly build AI-powered applications. Additionally, Accenture will establish a center of excellence focused on MongoDB projects, being the first global systems integrator to join MAAP. MongoDB 8.0 was also announced, featuring significant performance improvements such as faster reads and updates, along with significantly faster bulk inserts and time series queries. The general availability of Atlas Stream Processing was also announced, allowing users to build sophisticated, event-driven applications with real-time data. These announcements demonstrate MongoDB's expansion and focus on AI-driven application development.


MongoDB's latest AI announcements include the introduction of the MongoDB AI Applications Program (MAAP), which provides customers with reference architectures, pre-built partner integrations, and professional services to help them quickly build AI-powered applications. Additionally, Accenture will establish a center of excellence focused on MongoDB projects, being the first global systems integrator to join MAAP. MongoDB 8.0 was also announced, featuring significant performance improvements such as faster reads and updates, along with significantly faster bulk inserts and time series queries. The general availability of Atlas Stream Processing was also announced, allowing users to build sophisticated, event-driven applications with real-time data. These announcements demonstrate MongoDB's expansion and focus on AI-driven application development.